# Business Understanding

## Background

* The AIC Kijabe hospital's outpatient department (OPD) currently manages patient flow across approximately forty departments (General OPD, Casualty, Renal, Oncology, and others) without a structured way to anticipate periods of high patient volume. This can lead to under-staffing during surges and inefficient resource allocation during quieter periods.

## Business Objectives

* Enable the hospital to anticipate periods of higher patient volume in advance, so that staffing and resources can be planned proactively rather than reactively.
* Understand whether seasonal patterns (i.e rainy vs dry seasons in Kenya) affect patient volume, to support longer-term capacity planning.
* Understand patient return behavior (how long patients typically go before returning to the hospital) to support follow-up care planning and identify departments with outlying short or long return intervals.

## Data Analysis Goals

* Build a time series forecasting model to predict daily department-level patient arrival volume.
* Quantify the relationship between seasonal patterns and patient volume, including any lagged effects.
* Apply survival analysis to model time-to-return-visit, since discharge/exit data is not available in this dataset, return-visit interval is used as a signal for patient care continuity.

## Success Criteria

* Surge model - Forecast accuracy to be more useful than a simple "same as last week" baseline (e.g., a measurable improvement in MAE/RMSE).
* Seasonal correlation - A clear, evidence-based statement on whether seasons have a meaningful effect on patient volume.
* Survival analysis - Identifiable differences in return-visit patterns across at least one patient segment (e.g., department, age group).

# Data Preparation

In [19]:
# %pip install openpyxl

In [20]:
# import the necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

In [21]:
# load the data
data = pd.read_excel("Opd_data.xlsx")
data.head()

,PatientNumber,RegistrationDate,Gender,Age,QueuedTo,ConsultDescription
0,010575758,2024-06-01 00:00:00.000,Male,38 Yr(s),GENERAL OPD,GEneral Outpatient Care( NEW )
1,010575713,2024-06-01 00:00:00.000,Female,29 Yr(s),NaN,NaN
2,010569291,2024-06-01 00:04:38.083,Female,44 Yr(s),GENERAL OPD,General Outpatient Care
3,010575732,2024-06-01 00:04:42.460,Female,78 Yr(s),GENERAL OPD,General Outpatient Care
4,010575731,2024-06-01 00:10:16.037,Female,2 Yr(s),ADMISSION,Admission


In [22]:
# Basic exploratory
data.shape
print(data.info())

<class 'pandas.DataFrame'>
RangeIndex: 306306 entries, 0 to 306305
Data columns (total 6 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   PatientNumber       306306 non-null  str           
 1   RegistrationDate    306306 non-null  datetime64[us]
 2   Gender              306291 non-null  str           
 3   Age                 305735 non-null  str           
 4   QueuedTo            299528 non-null  str           
 5   ConsultDescription  299920 non-null  str           
dtypes: datetime64[us](1), str(5)
memory usage: 14.0 MB
None


## Data Cleaning

In [23]:
# Check for missing values
data.isnull().sum()

PatientNumber            0
RegistrationDate         0
Gender                  15
Age                    571
QueuedTo              6778
ConsultDescription    6386
dtype: int64

In [24]:
# check for duplicates
data.duplicated().sum()

355

In [25]:
# Drop the duplicates
data= data.drop_duplicates()

In [26]:
# Recheck for duplicates
data.duplicated().sum()

0

In [27]:
# Drop gender null values
data = data[data['Gender'].notna()]

In [28]:
# Standardize casing
data['Gender'] = data['Gender'].str.strip().str.upper()